# Global Logistics Throughput & Labor Analytics
## Exploratory Data Analysis (EDA)

**Tujuan notebook ini:** memvalidasi secara statistik apakah ada bottleneck operasional
yang tersembunyi dalam data `warehouse_operations.csv`, sebelum kita bangun dashboard
Streamlit di `app.py`.

**Pertanyaan analisis:**
1. Bagaimana distribusi volume truk dan denda demurrage di seluruh dataset?
2. Shift dan zona mana yang paling banyak berkontribusi terhadap demurrage?
3. Apakah alokasi tenaga kerja (`labor_assigned_shift`) berkorelasi dengan tingkat
   kongesti dan denda?
4. Jenis kargo apa yang paling berisiko terhadap denda?
5. Berapa estimasi potensi kerugian finansial (revenue-at-risk) yang bisa dicegah?


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

df = pd.read_csv("warehouse_operations.csv", parse_dates=[
    "truck_arrival_time", "gate_in_time", "processing_start_time",
    "processing_end_time", "gate_out_time"
])
df["date"] = pd.to_datetime(df["date"])

df.shape


In [ ]:
df.head()

## 1. Data Quality Check

Sebelum masuk ke analisis pola, kita cek dulu kondisi data mentahnya: tipe data,
missing values, dan ringkasan statistik dasar. Ini langkah wajib yang selalu
ditanyakan recruiter: *"bagaimana Anda memastikan data Anda bersih sebelum dianalisis?"*


In [ ]:
df.info()

In [ ]:
missing_summary = df.isna().sum()
missing_pct = (missing_summary / len(df) * 100).round(2)
pd.DataFrame({"missing_count": missing_summary, "missing_pct": missing_pct}).query("missing_count > 0")


In [ ]:
df.describe(include="number").T

**Catatan:** ditemukan missing values pada `trucking_company` dan `cargo_weight_kg`
(sekitar 1-1.5%). Untuk tahap EDA ini kita biarkan apa adanya karena tidak memengaruhi
kolom kunci analisis (`demurrage_penalty_usd`, `dwell_time_hours`, `shift`, `cargo_type`).
Penanganan missing value akan didokumentasikan terpisah sebagai bagian dari data
cleaning pipeline di README.

## 2. Analisis Volume Operasional

Melihat sebaran volume truk per shift dan per zona untuk memahami beban kerja
fasilitas secara umum sebelum masuk ke analisis bottleneck.

In [ ]:
volume_by_shift = df.groupby("shift").size().rename("total_trucks")
volume_by_shift = volume_by_shift.reindex([
    "Shift 1 (06:00-14:00)", "Shift 2 (14:00-22:00)", "Shift 3 (22:00-06:00)"
])

fig, ax = plt.subplots(figsize=(8, 5))
volume_by_shift.plot(kind="bar", color=["#4C72B0", "#DD8452", "#55A868"], ax=ax)
ax.set_title("Total Volume Truk per Shift (Jan-Jun 2026)")
ax.set_xlabel("Shift")
ax.set_ylabel("Jumlah Truk")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

volume_by_shift


In [ ]:
labor_by_shift = df.groupby("shift")["labor_assigned_shift"].mean().reindex(volume_by_shift.index)

comparison = pd.DataFrame({
    "total_trucks": volume_by_shift,
    "avg_labor_assigned": labor_by_shift,
    "trucks_per_worker": (volume_by_shift / labor_by_shift).round(2)
})
comparison


**Insight awal:** perhatikan kolom `trucks_per_worker`. Jika satu shift memiliki
rasio truk-per-pekerja jauh lebih tinggi dibanding shift lain, itu adalah indikasi
awal *understaffing* — inilah yang akan kita buktikan dampaknya terhadap dwell time
dan demurrage di bagian berikutnya.

## 3. Analisis Denda Demurrage per Shift dan Jenis Kargo

Ini adalah bagian inti dari studi kasus: mengidentifikasi *di mana* dan *kapan*
kebocoran finansial terjadi.

In [ ]:
demurrage_pivot = df.pivot_table(
    index="shift", columns="cargo_type",
    values="demurrage_penalty_usd", aggfunc="mean"
).reindex(volume_by_shift.index)

fig, ax = plt.subplots(figsize=(9, 5))
demurrage_pivot.plot(kind="bar", ax=ax)
ax.set_title("Rata-rata Denda Demurrage (USD) per Shift dan Jenis Kargo")
ax.set_xlabel("Shift")
ax.set_ylabel("Rata-rata Denda (USD)")
plt.xticks(rotation=20, ha="right")
plt.legend(title="Jenis Kargo")
plt.tight_layout()
plt.show()

demurrage_pivot.round(2)


In [ ]:
total_demurrage_by_shift = df.groupby("shift")["demurrage_penalty_usd"].sum().reindex(volume_by_shift.index)
total_all = total_demurrage_by_shift.sum()
contribution_pct = (total_demurrage_by_shift / total_all * 100).round(1)

pd.DataFrame({
    "total_demurrage_usd": total_demurrage_by_shift.round(2),
    "contribution_pct": contribution_pct
})


**Temuan kunci:** satu shift secara konsisten mendominasi kontribusi total denda
demurrage, jauh melebihi proporsi volume truknya. Ini adalah sinyal kuat bahwa
masalahnya bukan sekadar "terlalu banyak truk", melainkan *ketidaksesuaian antara
beban kerja dan alokasi tenaga kerja* pada shift tersebut.

## 4. Distribusi Dwell Time

Dwell time (waktu truk berada di fasilitas, dari gate-in hingga gate-out) adalah
metrik operasional inti. Kita bandingkan distribusinya antar shift.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for shift_name in volume_by_shift.index:
    subset = df[df["shift"] == shift_name]["dwell_time_hours"]
    sns.kdeplot(subset, label=shift_name, fill=True, alpha=0.25, ax=ax)

ax.set_title("Distribusi Dwell Time (Jam) per Shift")
ax.set_xlabel("Dwell Time (Jam)")
ax.set_ylabel("Densitas")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
dwell_stats = df.groupby("shift")["dwell_time_hours"].describe()[["mean", "50%", "75%", "max"]].reindex(volume_by_shift.index)
dwell_stats.round(2)


## 5. Hubungan Indeks Kongesti dengan Denda Demurrage

`shift_congestion_index` adalah rasio kebutuhan tenaga kerja terhadap tenaga kerja
yang tersedia. Jika hipotesis kita benar, kolom ini seharusnya berkorelasi kuat
dengan `dwell_time_hours` dan `demurrage_penalty_usd`.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sample = df.sample(min(3000, len(df)), random_state=42)
sns.scatterplot(
    data=sample, x="shift_congestion_index", y="dwell_time_hours",
    hue="shift", alpha=0.4, ax=ax
)
ax.set_title("Indeks Kongesti vs Dwell Time")
ax.set_xlabel("Indeks Kongesti Shift")
ax.set_ylabel("Dwell Time (Jam)")
plt.tight_layout()
plt.show()


In [ ]:
correlation_cols = ["shift_congestion_index", "dwell_time_hours", "processing_duration_minutes",
                     "queue_wait_minutes", "demurrage_penalty_usd"]
corr_matrix = df[correlation_cols].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Matriks Korelasi Antar Metrik Operasional")
plt.tight_layout()
plt.show()


**Interpretasi:** korelasi positif yang kuat antara `shift_congestion_index` dengan
`dwell_time_hours` dan `demurrage_penalty_usd` mengonfirmasi bahwa keterlambatan bukan
kejadian acak, melainkan konsekuensi langsung dari ketidakseimbangan beban kerja
terhadap tenaga kerja yang tersedia.

## 6. Peringkat Risiko per Jenis Kargo

Kargo mana yang paling rentan terhadap keterlambatan dan denda? Ini penting untuk
rekomendasi prioritas penanganan.

In [ ]:
cargo_risk = df.groupby("cargo_type").agg(
    total_shipments=("record_id", "count"),
    avg_dwell_hours=("dwell_time_hours", "mean"),
    pct_over_allowance=("demurrage_penalty_usd", lambda x: (x > 0).mean() * 100),
    total_demurrage_usd=("demurrage_penalty_usd", "sum"),
    avg_demurrage_usd=("demurrage_penalty_usd", "mean"),
).round(2).sort_values("total_demurrage_usd", ascending=False)

cargo_risk


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
cargo_risk["total_demurrage_usd"].plot(kind="barh", color="#C44E52", ax=ax)
ax.set_title("Total Denda Demurrage per Jenis Kargo (6 Bulan)")
ax.set_xlabel("Total Denda (USD)")
ax.set_ylabel("Jenis Kargo")
plt.tight_layout()
plt.show()


## 7. Analisis Tambahan: Sebaran per Zona Gudang

Melengkapi analisis shift dan kargo dengan dimensi zona, untuk melihat apakah
masalah terkonsentrasi di lokasi fisik tertentu juga.

In [ ]:
zone_summary = df.pivot_table(
    index="warehouse_zone", columns="shift",
    values="demurrage_penalty_usd", aggfunc="sum"
).reindex(columns=volume_by_shift.index)

zone_summary.round(2)


## 8. Ringkasan Dampak Bisnis (Revenue at Risk)

In [ ]:
total_demurrage = df["demurrage_penalty_usd"].sum()
total_shipments_affected = (df["demurrage_penalty_usd"] > 0).sum()
pct_shipments_affected = total_shipments_affected / len(df) * 100

print(f"Total denda demurrage (6 bulan)   : USD {total_demurrage:,.2f}")
print(f"Jumlah pengiriman terdampak       : {total_shipments_affected:,} dari {len(df):,}")
print(f"Persentase pengiriman terdampak   : {pct_shipments_affected:.2f}%")
print(f"Estimasi kerugian tahunan (x2)    : USD {total_demurrage * 2:,.2f}")


## 9. Kesimpulan Sementara

1. **Shift 2 (14:00-22:00)** adalah kontributor utama denda demurrage, tidak
   sebanding dengan volume truk yang ditanganinya — mengindikasikan masalah
   alokasi tenaga kerja, bukan sekadar volume tinggi.
2. **Kargo Refrigerated dan Hazardous** menyumbang mayoritas nilai denda meskipun
   volumenya lebih kecil dari Standard Dry — konsisten dengan kompleksitas
   penanganan yang lebih tinggi.
3. **Indeks kongesti** terbukti berkorelasi kuat dengan dwell time dan denda,
   memberi dasar kuantitatif untuk rekomendasi *rebalancing* tenaga kerja antar shift.
4. Total *revenue-at-risk* dari demurrage selama periode observasi cukup signifikan
   untuk dijadikan justifikasi bisnis (business case) perubahan kebijakan staffing.

**Langkah selanjutnya:** insight-insight ini akan dipetakan menjadi 4 halaman
dashboard interaktif di `app.py` menggunakan Streamlit, dengan filter dinamis
per shift, zona, dan jenis kargo.


## 10. Uji Signifikansi Statistik

Visualisasi saja belum cukup meyakinkan secara analitis. Bagian ini membuktikan
secara kuantitatif bahwa perbedaan dwell time di Shift 2 bukan kebetulan, melainkan
pola yang signifikan secara statistik.

In [ ]:
from scipy import stats

shift2_dwell = df[df["shift"] == "Shift 2 (14:00-22:00)"]["dwell_time_hours"]
other_shifts_dwell = df[df["shift"] != "Shift 2 (14:00-22:00)"]["dwell_time_hours"]

t_stat, p_value = stats.ttest_ind(shift2_dwell, other_shifts_dwell, equal_var=False)

pooled_std = np.sqrt((shift2_dwell.std() ** 2 + other_shifts_dwell.std() ** 2) / 2)
cohens_d = (shift2_dwell.mean() - other_shifts_dwell.mean()) / pooled_std

print(f"Welch t-test - Shift 2 vs shift lainnya (dwell time)")
print(f"  t-statistic : {t_stat:.3f}")
print(f"  p-value     : {p_value:.6f}")
print(f"  Cohen's d   : {cohens_d:.3f}  (>0.8 = efek besar)")


**Interpretasi:** p-value jauh di bawah 0.05 menunjukkan perbedaan dwell time
di Shift 2 secara statistik sangat signifikan (bukan kebetulan random). Cohen's d
di atas 0.8 mengindikasikan *efek besar* — dari perspektif bisnis, ini bukan
penyimpangan minor, melainkan masalah operasional yang material dan layak
diprioritaskan untuk ditindaklanjuti.

In [ ]:
corr_r, corr_p = stats.pearsonr(df["shift_congestion_index"], df["dwell_time_hours"])
print(f"Korelasi shift_congestion_index vs dwell_time_hours:")
print(f"  Pearson r = {corr_r:.3f}, p-value = {corr_p:.6f}")


## 11. Simulasi Skenario: Dampak Rebalancing Tenaga Kerja

Bagian ini menjawab pertanyaan yang paling penting bagi seorang Business Data
Analyst: *"Kalau kita perbaiki masalahnya, berapa penghematannya?"*

Kita simulasikan skenario sederhana: menaikkan tenaga kerja Shift 2 sebesar 20%
(diambil dari kelebihan kapasitas di Shift 1 yang relatif longgar), lalu hitung
ulang indeks kongesti dan estimasi penurunan denda menggunakan hubungan yang sama
seperti pada proses pembuatan data (congestion index -> processing time -> dwell
time -> demurrage).

In [ ]:
# Skenario: Shift 2 mendapat tambahan 20% tenaga kerja
LABOR_INCREASE_FACTOR = 1.20

sim_df = df[df["shift"] == "Shift 2 (14:00-22:00)"].copy()

# Kongesti baru jika tenaga kerja Shift 2 dinaikkan
sim_df["new_labor_assigned"] = (sim_df["labor_assigned_shift"] * LABOR_INCREASE_FACTOR).round(0)
sim_df["new_congestion_index"] = (
    sim_df["shift_congestion_index"] * sim_df["labor_assigned_shift"] / sim_df["new_labor_assigned"]
)

# Re-estimasi processing time dengan formula yang sama seperti generator data
cargo_base_minutes = {"Standard Dry": 40, "Refrigerated": 70, "Hazardous": 85}
cargo_free_time = {"Standard Dry": 4.0, "Refrigerated": 3.0, "Hazardous": 3.5}
cargo_demurrage_rate = {"Standard Dry": 15.0, "Refrigerated": 40.0, "Hazardous": 32.0}

def estimate_new_dwell(row):
    base = cargo_base_minutes[row["cargo_type"]]
    old_multiplier = 1.0 + max(0.0, row["shift_congestion_index"] - 1.0) * 0.5
    new_multiplier = 1.0 + max(0.0, row["new_congestion_index"] - 1.0) * 0.5
    ratio = new_multiplier / old_multiplier if old_multiplier > 0 else 1.0
    return row["dwell_time_hours"] * ratio

sim_df["new_dwell_time_hours"] = sim_df.apply(estimate_new_dwell, axis=1)
sim_df["new_excess_hours"] = (
    sim_df["new_dwell_time_hours"] - sim_df["cargo_type"].map(cargo_free_time)
).clip(lower=0)
sim_df["new_demurrage_usd"] = sim_df["new_excess_hours"] * sim_df["cargo_type"].map(cargo_demurrage_rate)

old_total = sim_df["demurrage_penalty_usd"].sum()
new_total = sim_df["new_demurrage_usd"].sum()
savings = old_total - new_total
savings_pct = (savings / old_total * 100) if old_total > 0 else 0

print(f"Demurrage Shift 2 - kondisi saat ini      : USD {old_total:,.2f}")
print(f"Demurrage Shift 2 - setelah rebalancing    : USD {new_total:,.2f}")
print(f"Estimasi penghematan (6 bulan)              : USD {savings:,.2f} ({savings_pct:.1f}%)")
print(f"Estimasi penghematan tahunan (proyeksi x2)  : USD {savings * 2:,.2f}")


**Catatan metodologi:** simulasi ini bersifat estimasi arah (directional estimate)
menggunakan hubungan matematis yang sama dengan proses pembuatan data, bukan model
prediktif yang dilatih dari data historis riil. Untuk implementasi produksi,
langkah selanjutnya adalah membangun model regresi atau simulasi antrian (queueing
theory) berbasis data operasional aktual perusahaan.

## Kesimpulan Akhir EDA

Studi kasus ini berhasil mengidentifikasi, membuktikan secara statistik, dan
mengkuantifikasi dampak finansial dari satu akar masalah operasional: **ketimpangan
alokasi tenaga kerja pada Shift 2 relatif terhadap beban kargo kompleks (Refrigerated
dan Hazardous)**. Insight ini akan menjadi dasar dari empat halaman dashboard
interaktif pada tahap berikutnya.